In [40]:
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
import operator

In [41]:
class OutputSchema(BaseModel):
    feedback : str = Field(description="give the feedback of essay")
    score : int = Field(description="provide the essay score form 1 to 10", ge=1, le=10)


In [42]:
llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    temperature = 0.8
)


In [43]:
structured_model = llm.with_structured_output(OutputSchema)

In [44]:
class EssayEvalution(TypedDict):
    essay : str
    quality_feedback : str
    depth_feedback : str
    language_feedback : str
    overall_feedback : str
    individual_score : Annotated[list[int], operator.add]
    avg_score : float
    


In [45]:
Essay = "Technology is very important in our daily life. We are using technology everywhere and every time. It is helping people in many ways, but sometimes it is also creating problems. Technology is a good thing because it makes our work easy, but sometimes people are using it too much and they are not thinking about other things. Nowadays, mobile phones are used by almost everyone. People use mobile phones for calling, watching videos, playing games, studying and many other things. But some people are spending too much time on mobile phones. They are not using their time properly. This is not good for their life, but still they are doing it because mobile phones are interesting. Technology is also helping students in education. Students can learn many things from the internet. They can watch videos and read information. But sometimes the information is not correct, and students may get confused. Also, students are depending on technology instead of thinking by themselves. This can become a problem in the future. In offices, technology is making work faster. Computers are doing many tasks and saving time. But some people are worried that technology will take their jobs. This is also a serious issue because everyone needs a job to live a good life. Another thing is that technology is changing very fast. People need to learn new things again and again. Sometimes it is difficult to understand everything. But we cannot stop technology because it is becoming part of our life. In conclusion, technology is important, but we should use it carefully. It has good things and bad things. We should understand both sides and use technology in a proper way. Technology is helping us, but we should also think about our own abilities and responsibilities."

In [46]:
def quality_content(state : EssayEvalution):
    essay = state["essay"]
    prompt = f""" you are a Essay analaysis Expect, analyse the essay based on its clarity of thought and give the feedback and also provide the score from range 1 to 10 {essay} """
    response = structured_model.invoke(prompt)
    return {
        "quality_feedback" : response.feedback,
        "individual_score" : [response.score]
    }

In [47]:
def depth_content(state : EssayEvalution):
    essay = state["essay"]
    prompt = f""" you are a Essay analaysis Expect, analyse the essay based on its depth and give the feedback and also provide the score from range 1 to 10 {essay} """
    response = structured_model.invoke(prompt)
    return {
        "depth_feedback" : response.feedback,
        "individual_score" : [response.score]
    }

In [48]:
def language_content(state : EssayEvalution):
    essay = state["essay"]
    prompt = f""" you are a Essay analaysis Expect, analyse the essay based on its language grammar and usage and give the feedback and also provide the score from range 1 to 10 {essay} """
    response = structured_model.invoke(prompt)
    return {
        "language_feedback" : response.feedback,
        "individual_score" : [response.score]
    }

In [49]:
def overall_analysis(state : EssayEvalution):
    essay = state["essay"]
    quality_feedback = state["quality_feedback"]
    depth_feedback = state["depth_feedback"]
    language_feedback = state["language_feedback"]
    prompt = f""" you are an high professional essay evaluator analayse the all three feedback form the {quality_feedback}, {depth_feedback}, {language_feedback} and provide the overall feedback on the essay"""
    response = llm.invoke(prompt)
    average_score = sum(state["individual_score"])/len(state["individual_score"])

    return {
        "overall_feedback" : response.content,
        "avg_score" : average_score
    }

In [50]:
graph = StateGraph(EssayEvalution)

graph.add_node("quality_content", quality_content)
graph.add_node("depth_content", depth_content)
graph.add_node("language_content", language_content)
graph.add_node("overall_analysis", overall_analysis)

analysis_node = ["quality_content", "depth_content", "language_content"]

for node in analysis_node:
    graph.add_edge(START, node)

graph.add_edge(analysis_node, "overall_analysis")
graph.add_edge("overall_analysis", END)

workflow = graph.compile()

In [52]:
initial_state = {
    "essay" : Essay
}
final_state = workflow.invoke(initial_state)
print(final_state)

{'essay': 'Technology is very important in our daily life. We are using technology everywhere and every time. It is helping people in many ways, but sometimes it is also creating problems. Technology is a good thing because it makes our work easy, but sometimes people are using it too much and they are not thinking about other things. Nowadays, mobile phones are used by almost everyone. People use mobile phones for calling, watching videos, playing games, studying and many other things. But some people are spending too much time on mobile phones. They are not using their time properly. This is not good for their life, but still they are doing it because mobile phones are interesting. Technology is also helping students in education. Students can learn many things from the internet. They can watch videos and read information. But sometimes the information is not correct, and students may get confused. Also, students are depending on technology instead of thinking by themselves. This can